# GEOG 464 starter pipeline: CSV to GeoJSON

This notebook is the data link of the course pipeline: **CSV -> Python -> GeoJSON -> Leaflet**.

```
geog464-starter/
  data/sample_places.csv     input table (edit this)
  data/places.geojson        output, written by this notebook
  notebooks/mvp_pipeline.ipynb
  site/index.html, style.css, app.js
```

Run each cell in order, top to bottom. Cell 3 writes `data/places.geojson`, which the web page in `site/` draws. If a cell fails with NameError, run the cells above it first (or use Run All).


## 1) Import libraries and load the CSV
We start by loading `../data/sample_places.csv`. For your project, replace it with your own CSV (must include `lat` and `lon`).

In [6]:
import pandas as pd
csv_path = "../data/sample_places.csv"
df = pd.read_csv(csv_path)
df.head()

,name,category,description,lat,lon
0,Mount Royal Chalet,Landmark,Lookout over downtown at the top of the mountain,45.5037,-73.5872
1,THomas,Food,Open-air public market in Little Italy,45.5364,-73.6144
2,Atwater Market,Food,Market hall beside the Lachine Canal,45.4794,-73.5772
3,Parc La Fontaine,Nature,Park with two ponds in the Plateau,45.5250,-73.5690
4,Concordia Hall Building,Landmark,Where GEOG 464 meets,45.4973,-73.5789


## 2) (Optional) Quick filtering/cleaning
For MVP, just loading is enough. Example filter: keep only rows whose `category` is `Landmark`. This cell only displays a filtered copy; it does not change `df`, so the export below still writes every row. Lab 2 is where filtering becomes yours.

In [7]:
landmarks = df[df["category"] == "Landmark"]
landmarks

,name,category,description,lat,lon
0,Mount Royal Chalet,Landmark,Lookout over downtown at the top of the mountain,45.5037,-73.5872
4,Concordia Hall Building,Landmark,Where GEOG 464 meets,45.4973,-73.5789


## 3) Convert DataFrame to GeoJSON
We’ll iterate through rows and build a GeoJSON `FeatureCollection`. **Output:** `../data/places.geojson`

In [8]:
import json
features = []
for _, row in df.iterrows():
    try:
        lat = float(row["lat"])
        lon = float(row["lon"])
    except Exception:
        continue  # skip bad rows
    feature = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lon, lat]},
        "properties": {
            "name": str(row.get("name", "")).strip(),
            "category": str(row.get("category", "")).strip(),
            "description": str(row.get("description", "")).strip()
        }
    }
    features.append(feature)
geojson = {"type": "FeatureCollection", "features": features}
out_path = "../data/places.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)
print(f"GeoJSON created with {len(features)} features, written to {out_path}")

GeoJSON created with 5 features, written to ../data/places.geojson


## 4) Preview the GeoJSON
Read the file back in and show the first feature.

In [9]:
with open("../data/places.geojson", "r", encoding="utf-8") as f:
    data = json.load(f)
print("Features:", len(data.get("features", [])))
data["features"][0] if data.get("features") else {}

Features: 5


{'type': 'Feature',
 'geometry': {'type': 'Point', 'coordinates': [-73.5872, 45.5037]},
 'properties': {'name': 'Mount Royal Chalet',
  'category': 'Landmark',
  'description': 'Lookout over downtown at the top of the mountain'}}

## 5) Next: Connect to Leaflet
1. In VS Code, right-click `site/index.html` and choose Open with Live Server.
2. `site/app.js` fetches `../data/places.geojson` and adds it to the map with `L.geoJSON(...)`.